# Enterprise Track 1 · Unity Catalog: a Home for the Data

**Goal:** build the governed place the rest of the track uses: your schema, a landing volume with the first source delivery in it, and the crosswalks from session 2 as governed reference tables. Along the way: three-part names, ownership, comments, tags and grants.

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 1

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Unity Catalog")

## 1. The object model in one picture
```
metastore (one per region)
└── catalog            e.g. dmo_sandbox            USE CATALOG
    └── schema         e.g. dmo_sandbox.lnl_you     USE SCHEMA, CREATE TABLE, CREATE VOLUME ...
        ├── table      bronze_offenders, ref_race_crosswalk    SELECT, MODIFY
        ├── view       offenders_analyst                       SELECT
        ├── volume     landing (files, not rows)               READ VOLUME, WRITE VOLUME
        └── function   mask_restricted                         EXECUTE
```
Every object has one **owner** (you, for everything you create here) and **grants**. Privileges flow down: `SELECT` granted on a schema covers every table in it, now and later. A reader also needs `USE CATALOG` and `USE SCHEMA` on the containers.

**Managed** tables and volumes (what this track uses) keep their files in storage Unity Catalog manages; drop the object and the data goes too. **External** ones point at storage someone else manages, such as an existing S3 bucket. Prefer managed unless you have a reason.

In [ ]:
ent.sql(f"CREATE SCHEMA IF NOT EXISTS {N['fq']} COMMENT 'Lunch & Learn enterprise track (mock data only)'")
ent.set_tags(f"ALTER SCHEMA {N['fq']} SET TAGS ('course' = 'lunch-and-learn', 'data' = 'mock')")
ent.sql(f"CREATE VOLUME IF NOT EXISTS {N['volume_name']} COMMENT 'Landing zone: source files arrive here'")
display(ent.sql(f"DESCRIBE SCHEMA EXTENDED {N['fq']}"))

## 2. A landing zone: files in a volume
A **volume** holds files, not rows: it's the governed landing zone from the handout's "How data moves" page. Source systems drop files here; the pipeline in notebook 02 picks up each new file once. The volume is a path anyone with access can use like a folder: `/Volumes/<catalog>/<schema>/<volume>/`.

This cell plays the source system and delivers the first batch: the course's 100,000 records, exactly as the session notebooks get them.

In [ ]:
ent.land_full_dataset(N)
for f in sorted(Path(N["landing"]).iterdir()):
    print(f"{f.name:20} {f.stat().st_size / 1_048_576:6.1f} MB")

In [ ]:
# Peek at the delivery with every column as text, the way session 2 loads it
peek = spark.read.options(header=True, inferSchema=False).csv(N["landing"])
print(f"{peek.count():,} rows, {len(peek.columns)} columns, all {set(t for _, t in peek.dtypes)}")
display(peek.select("tdcj_number", "unit_location", "race", "stg_affiliation", "offense_code").limit(5))

## 3. Reference data as governed tables
In session 2 the crosswalks were CSV files in Git. That's still where they're **edited and reviewed**. In production they're also loaded into governed tables, so pipelines, reports and other teams all read the same agreed version, with an owner, a comment, grants and lineage. When a data steward approves a change, the CSV changes in Git and this load runs again.

In [ ]:
from lnl import domains

descriptions = {
    "race_crosswalk": "Race as written -> agency letter code and 2024 federal (OMB) category",
    "religion_crosswalk": "Religion as written -> faith group and family",
    "stg_groups": "STG alias as written -> fictional group name",
    "nibrs_offenses": "FBI NIBRS offense codes used in the course and what each crime is against",
    "violence_codes": "Violence code as written -> NV, V1, V2, V3",
    "unit_crosswalk": "Unit as written (name, legacy code, variants) -> unit name and 3-letter code",
}
for name in ent.REFERENCE_TABLES:
    table = f"{N['fq']}.ref_{name}"
    (spark.createDataFrame(domains.reference(name))
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table))
    ent.sql(f"COMMENT ON TABLE {table} IS '{descriptions[name]}. Source: lnl/reference/{name}.csv in Git.'", quiet=True)
    ent.set_tags(f"ALTER TABLE {table} SET TAGS ('data_kind' = 'reference', 'source' = 'git')")
    print(f"{table:60} {spark.table(table).count():>4} rows")

In [ ]:
display(ent.sql(f"""
    SELECT table_name, table_type, comment
    FROM {N['catalog']}.information_schema.tables
    WHERE table_schema = '{N['schema']}'
    ORDER BY table_name"""))

`information_schema` is SQL-standard metadata about everything in a catalog: tables, columns, tags, grants. It answers "what do we have, and what does it mean?" with a query instead of a meeting.

## 4. Grants
You own everything you just created, so you can grant access to it. Reference data isn't sensitive, so analysts can read all of it. A reader needs three things: `USE CATALOG`, `USE SCHEMA`, and `SELECT`. If `ANALYST_GROUP` isn't set, the statements are printed instead of run.

In [ ]:
group = config.ANALYST_GROUP
statements = [f"GRANT USE SCHEMA ON SCHEMA {N['fq']} TO `{group or '<analyst group>'}`"] + [
    f"GRANT SELECT ON TABLE {N['fq']}.ref_{name} TO `{group or '<analyst group>'}`" for name in ent.REFERENCE_TABLES]
for statement in statements:
    if group:
        ent.sql(statement)
    else:
        print(statement + ";   -- not run: set ANALYST_GROUP in lnl/config.py")
print(f"\nThe group also needs USE CATALOG on {N['catalog']}, which the catalog's owner grants.")
display(ent.sql(f"SHOW GRANTS ON SCHEMA {N['fq']}"))

## Try it
- Open **Catalog** in the left sidebar and find your schema. Every comment and tag you set is there, and the **Permissions** tab shows the grants.
- `SELECT * FROM <your schema>.ref_unit_crosswalk WHERE unit_name = 'Gotham City'` lists every agreed spelling of one unit.

**Next:** 02_pipeline turns the landed file into bronze, silver and gold tables, with every session 2-4 rule enforced as an expectation.